# Cryptocurrency 234 Coins Altcoins Prices

El dataset contiene precios históricos de 234 criptomonedas (incluyendo Bitcoin y altcoins), con los siguientes campos para cada transacción:
•	Open (precio de apertura)

•	High (precio máximo)

•	Low (precio mínimo)

•	Close (precio de cierre)

•	Volume (volumen operado)


Estos datos están disponibles en diferentes marcos de tiempo: W1 (Semanal), D1 (Diaria), H4 (4 horas), H1 (1 hora), M30 (30 minutos), M15 (15 minutos) y M5 (cinco minutos).


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,mean, avg, min, max, count, lit, round, concat_ws
import pandas as pd

In [3]:
#lectura del archivo
spark = SparkSession.builder.master("local[*]").getOrCreate()
df = spark.read.csv("/Users/salvadorhernandez/Documents/bigdata/dataset/dataset.csv", header=True, inferSchema=True)
df.show()

25/05/02 22:37:34 WARN Utils: Your hostname, MacBook-Pro-de-Salvador.local resolves to a loopback address: 127.0.0.1; using 192.168.0.150 instead (on interface en0)
25/05/02 22:37:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/02 22:37:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------------+--------+--------+--------+--------+---------+------+---------+
|           datetime|    open|    high|     low|   close|   volume|  coin|frequency|
+-------------------+--------+--------+--------+--------+---------+------+---------+
|2017-07-14 00:00:00|    0.08|0.091033|    0.08|0.090993| 1942.057|ETHBTC|       D1|
|2017-07-15 00:00:00|0.090993|0.093699|0.087127|0.087635| 4013.066|ETHBTC|       D1|
|2017-07-16 00:00:00|0.087508|0.087635|0.075591|0.082241| 8904.158|ETHBTC|       D1|
|2017-07-17 00:00:00|0.082368|0.088394|0.081699|0.087537| 6650.933|ETHBTC|       D1|
|2017-07-18 00:00:00|0.087831|0.109068|0.084777|0.107732| 7245.741|ETHBTC|       D1|
|2017-07-19 00:00:00|0.107732|0.108732| 0.08429|0.086853| 7384.976|ETHBTC|       D1|
|2017-07-20 00:00:00|0.087144|0.095252|0.074982|0.080509|  6395.16|ETHBTC|       D1|
|2017-07-21 00:00:00|0.080383|0.087511|0.079635|0.081581| 7434.389|ETHBTC|       D1|
|2017-07-22 00:00:00|0.081586|0.084733|0.079856|0.082474| 7979.89

In [32]:
#1. Caracterización de la población.

#principales caracteriscticas de variables númericas
numerical_stats = df.select(
    mean("open").alias("mean_open"), avg("open").alias("avg_open"), min("open").alias("min_open"), max("open").alias("max_open"),
    mean("high").alias("mean_high"), avg("high").alias("avg_high"), min("high").alias("min_high"), max("high").alias("max_high"),
    mean("low").alias("mean_low"), avg("low").alias("avg_low"), min("low").alias("min_low"), max("low").alias("max_low"),
    mean("close").alias("mean_close"), avg("close").alias("avg_close"), min("close").alias("min_close"), max("close").alias("max_close"),
    mean("volume").alias("mean_volume"), avg("volume").alias("avg_volume"), min("volume").alias("min_volume"), max("volume").alias("max_volume")
)

numerical_stats.show(truncate=False,vertical=True)


-RECORD 0-------------------------
 mean_open   | 315.30808366873765 
 avg_open    | 315.30808366873765 
 min_open    | 6.56E-5            
 max_open    | 90877.82           
 mean_high   | 316.24911175329464 
 avg_high    | 316.24911175329464 
 min_high    | 6.63E-5            
 max_high    | 95000.0            
 mean_low    | 314.33606158631443 
 avg_low     | 314.33606158631443 
 min_low     | 6.55E-5            
 max_low     | 90211.0            
 mean_close  | 315.3110150076161  
 avg_close   | 315.3110150076161  
 min_close   | 6.56E-5            
 max_close   | 90912.32           
 mean_volume | 3781457.4590252754 
 avg_volume  | 3781457.4590252754 
 min_volume  | 0.0                
 max_volume  | 3.326897912677E12  



In [33]:
# las 5 monedas principales

coin_counts = df.groupBy("coin").count().orderBy("count", ascending=False).limit(5).toPandas()
coin_counts.columns = ["coin", "coin_count"]
freq_counts = df.groupBy("frequency").count().orderBy("count", ascending=False).limit(5).toPandas()
freq_counts.columns = ["frequency", "freq_count"]

combined = pd.concat([coin_counts, freq_counts], axis=1)
print(combined)

      coin  coin_count frequency  freq_count
0   ETHBTC     1125355        M5    77826095
1  BTCUSDT     1109604       M15    25942097
2  ETHUSDT     1109604       M30    12971381
3  BNBUSDT     1072225        H1     6486216
4  NEOUSDT     1065744        H4     1622504


In [9]:
#Particonamiento
# Total de registros
combinaciones = [
    ("ETHBTC", "M5", 0.06),  # Combinación A -> a, B -> c
    ("BTCUSDT", "M15", 0.24),  # Combinación A -> a, B -> d
    ("ETHUSDT", "M30", 0.14),  # Combinación A -> b, B -> c
    ("BNBUSDT", "H1", 0.56)   # Combinación A -> b, B -> d
]

# **7. Recuperar submuestras según las reglas de particionamiento**

# Iterar sobre las combinaciones y filtrar los registros de acuerdo con las reglas definidas
for comb in combinaciones:
    a_value = comb[0]
    b_value = comb[1]
    probabilidad = comb[2]

    # Filtrar el DataFrame según la combinación
    particion = df.filter((col("coin") == a_value) & (col("frequency") == b_value))

    # Mostrar la partición filtrada y su probabilidad
    print(f"Combinación: Coin = {a_value}, Frequency = {b_value}, Probabilidad = {probabilidad}")
    particion.show()

    # Guardar la partición en un archivo CSV (opcional)
    particion.write.csv(f"output_{a_value}_{b_value}", header=True)

Combinación: Coin = ETHBTC, Frequency = M5, Probabilidad = 0.06


+-------------------+--------+--------+--------+--------+------+------+---------+
|           datetime|    open|    high|     low|   close|volume|  coin|frequency|
+-------------------+--------+--------+--------+--------+------+------+---------+
|2017-07-14 04:00:00|    0.08|    0.08|    0.08|    0.08| 0.726|ETHBTC|       M5|
|2017-07-14 04:05:00|    0.08|0.080001|    0.08|0.080001| 3.347|ETHBTC|       M5|
|2017-07-14 04:10:00|0.080001|  0.0864|0.080001|  0.0864| 4.679|ETHBTC|       M5|
|2017-07-14 04:15:00|0.085289| 0.08562|0.085128|0.085128|53.431|ETHBTC|       M5|
|2017-07-14 04:20:00|0.085274|   0.086|0.085274|   0.086| 5.576|ETHBTC|       M5|
|2017-07-14 04:25:00| 0.08525|0.085811| 0.08525|0.085811| 2.035|ETHBTC|       M5|
|2017-07-14 04:30:00|0.085811|   0.086|0.085811|   0.086|17.943|ETHBTC|       M5|
|2017-07-14 04:35:00| 0.08618| 0.08618| 0.08618| 0.08618|18.237|ETHBTC|       M5|
|2017-07-14 04:40:00| 0.08618| 0.08638| 0.08618|0.086314|17.589|ETHBTC|       M5|
|2017-07-14 04:4

Combinación: Coin = BTCUSDT, Frequency = M15, Probabilidad = 0.24


+-------------------+-------+-------+-------+-------+---------+-------+---------+
|           datetime|   open|   high|    low|  close|   volume|   coin|frequency|
+-------------------+-------+-------+-------+-------+---------+-------+---------+
|2017-08-17 04:00:00|4261.48|4280.56|4261.48|4261.48| 2.189061|BTCUSDT|      M15|
|2017-08-17 04:15:00|4261.48|4270.41|4261.32|4261.45| 9.119865|BTCUSDT|      M15|
|2017-08-17 04:30:00| 4280.0|4310.07|4267.99|4310.07|21.923552|BTCUSDT|      M15|
|2017-08-17 04:45:00|4310.07|4313.62|4291.37|4308.83|13.948531|BTCUSDT|      M15|
|2017-08-17 05:00:00|4308.83|4328.69|4304.31|4304.31| 5.101153|BTCUSDT|      M15|
|2017-08-17 05:15:00| 4320.0| 4320.0|4312.14| 4320.0|15.947495|BTCUSDT|      M15|
|2017-08-17 05:30:00| 4320.0| 4320.0|4291.37|4291.37| 2.155453|BTCUSDT|      M15|
|2017-08-17 05:45:00|4297.04|4315.32|4297.04|4315.32| 0.030815|BTCUSDT|      M15|
|2017-08-17 06:00:00|4330.29|4330.29|4318.39| 4330.0| 0.065364|BTCUSDT|      M15|
|2017-08-17 06:1

Combinación: Coin = ETHUSDT, Frequency = M30, Probabilidad = 0.14


+-------------------+------+------+------+------+---------+-------+---------+
|           datetime|  open|  high|   low| close|   volume|   coin|frequency|
+-------------------+------+------+------+------+---------+-------+---------+
|2017-08-17 04:00:00|301.13|301.13| 298.0|299.39| 37.24232|ETHUSDT|      M30|
|2017-08-17 04:30:00|299.39|302.57|299.39|301.61| 88.42645|ETHUSDT|      M30|
|2017-08-17 05:00:00|301.61|303.28| 300.0|302.21|231.80622|ETHUSDT|      M30|
|2017-08-17 05:30:00|302.21|303.28|302.08| 303.1|145.86624|ETHUSDT|      M30|
|2017-08-17 06:00:00| 302.4|303.71|302.02|303.11|117.26475|ETHUSDT|      M30|
|2017-08-17 06:30:00|303.11|304.44| 301.9|302.68|186.60197|ETHUSDT|      M30|
|2017-08-17 07:00:00|302.68|304.97| 302.6|304.61|437.24741|ETHUSDT|      M30|
|2017-08-17 07:30:00|304.61|307.96|304.53|307.96|317.49769|ETHUSDT|      M30|
|2017-08-17 08:00:00|307.95|309.77| 307.0|309.77| 95.56669|ETHUSDT|      M30|
|2017-08-17 08:30:00|309.77|309.97|307.86|308.62|  55.1836|ETHUS

Combinación: Coin = BNBUSDT, Frequency = H1, Probabilidad = 0.56
+-------------------+------+------+------+------+--------+-------+---------+
|           datetime|  open|  high|   low| close|  volume|   coin|frequency|
+-------------------+------+------+------+------+--------+-------+---------+
|2017-11-06 03:00:00|   1.5| 1.799|   0.5|   1.7|  649.12|BNBUSDT|       H1|
|2017-11-06 04:00:00|   1.3|  1.65|   1.3|1.6479| 8147.72|BNBUSDT|       H1|
|2017-11-06 05:00:00|1.5457|1.5525|1.5455|1.5458|  6628.2|BNBUSDT|       H1|
|2017-11-06 06:00:00|1.5458| 1.681|1.5387| 1.681| 22767.9|BNBUSDT|       H1|
|2017-11-06 07:00:00|1.6809|1.6809|   1.6| 1.625|14938.73|BNBUSDT|       H1|
|2017-11-06 08:00:00|1.6011|1.6379| 1.601|1.6378| 5888.89|BNBUSDT|       H1|
|2017-11-06 09:00:00| 1.602|1.6366|  1.58|1.6366| 6643.14|BNBUSDT|       H1|
|2017-11-06 10:00:00| 1.601|1.6479|   1.6|1.6479| 4885.62|BNBUSDT|       H1|
|2017-11-06 11:00:00|1.6479| 1.676|1.6288|1.6288| 4324.77|BNBUSDT|       H1|
|2017-11-06